# FordRetain — Análise Exploratória (Base 1)

Base real da Ford: **602.788 registros de manutenção · 175.554 veículos · 435 concessionárias · 2022–2025**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
sys.path.append('../src')
from preprocessamento import carregar_dados, construir_base1

os.makedirs('../outputs', exist_ok=True)

df = carregar_dados('../dados/vin_share_Desafio_02.xlsx')
print(f'Shape: {df.shape}')
print(f'VINs únicos: {df["VIN_Hash"].nunique()}')
print(f'Concessionárias: {df["DealerCode"].nunique()}')
print(f'Período: {df["ServiceDate"].min()} → {df["ServiceDate"].max()}')
df.head()

In [ ]:
print('Colunas disponíveis:')
for col in df.columns:
    print(f'  {col}: {df[col].dtype} — {df[col].nunique()} únicos — {df[col].isna().sum()} nulos')

In [ ]:
plt.figure(figsize=(12, 5))
df['ModelName'].value_counts().head(10).plot(kind='bar', color='#1F3A6E')
plt.title('Top 10 modelos — total de revisões na rede oficial')
plt.xlabel('Modelo')
plt.ylabel('Total de revisões')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/01_distribuicao_modelos.png', dpi=150)
plt.show()

In [ ]:
revisoes_por_vin = df.groupby('VIN_Hash')['MaintenanceNumber'].max()

plt.figure(figsize=(12, 5))
revisoes_por_vin.value_counts().sort_index().head(20).plot(kind='bar', color='#E74C3C')
plt.title('Quantas revisões cada veículo fez na rede oficial')
plt.xlabel('Número da última revisão')
plt.ylabel('Quantidade de veículos')
plt.tight_layout()
plt.savefig('../outputs/02_revisoes_por_vin.png', dpi=150)
plt.show()

total = revisoes_por_vin.count()
so_1 = (revisoes_por_vin == 1).sum()
print(f'Total de veículos: {total:,}')
print(f'Fizeram apenas 1 revisão e não voltaram: {so_1:,} ({so_1/total*100:.1f}%)')

In [ ]:
df['ano_revisao'] = df['ServiceDate'].dt.year
plt.figure(figsize=(10, 5))
df['ano_revisao'].value_counts().sort_index().plot(kind='bar', color='#27AE60')
plt.title('Volume de revisões por ano')
plt.xlabel('Ano')
plt.ylabel('Revisões')
plt.tight_layout()
plt.savefig('../outputs/03_revisoes_por_ano.png', dpi=150)
plt.show()

In [ ]:
base1 = construir_base1(df)
print(f'Base 1 shape: {base1.shape}')
base1.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(base1['total_revisoes'].clip(0, 10), bins=10, color='#1F3A6E', edgecolor='white')
axes[0].set_title('Distribuição: total de revisões por VIN')
axes[0].set_xlabel('Revisões')

axes[1].hist(base1['dias_desde_ultima_revisao'].dropna().clip(0, 500), bins=30, color='#E74C3C', edgecolor='white')
axes[1].set_title('Dias desde última revisão')
axes[1].set_xlabel('Dias')

axes[2].hist(base1['intervalo_medio_dias'].dropna().clip(0, 400), bins=30, color='#E67E22', edgecolor='white')
axes[2].set_title('Intervalo médio entre revisões (dias)')
axes[2].set_xlabel('Dias')

plt.tight_layout()
plt.savefig('../outputs/04_distribuicoes_base1.png', dpi=150)
plt.show()